- 锚定一个 script，继承基本的配置
    - https://github.com/verl-project/verl/blob/main/examples/grpo_trainer/run_qwen3_5_35b_megatron.sh
        - fsdp 不支持 EP
- 训练健康度检测
    - rollout_probs_diff_mean：1e-3
    - grad_norm
    - critic/score/mean
    - entropy
- 性能调优：5 steps 调参探测（probe）
    - 观测：timing
        - step = gen + ref/old logp + update_actor + update_weights(忽略不计)
            - ref log p: ref model 常驻GPU, ref.fsdp_config.param_offload=False
    - throughout：
        - perf/throughout：tok/s (每卡)，
            - log_prob_max_token_len_per_gpu *2/*3
            - dynamic bsz
- wandb 记录
    - val_before_train, eval: 5 steps, save_ckpt: 10 steps, wandb, 
    - 所有的run，都要记录必要的参数配置，方便做对比，以及调参经验的累积
- 更多的 GPUs 资源意味着什么
    - 更快的训练速度
    - 更多的参数性能调优的时间和机会

### wandb analysis

- model_dtype
    - https://wandb.ai/spatial-reasoning/geo3k_grpo?nw=nwuser1439217525
    - https://t.bilibili.com/1226574275331227666?spm_id_from=333.1387.0.0
- model size：qwen3.5-4b/9b, qwen3.6-35b-a3b
    - thinking model, response length, clip ratio
    - https://wandb.ai/1439217525-peking-university/geo3k_grpo_qwen35_4b?nw=nwuser1439217525
- rollout_correction
- rewards/scores, advantages/returns
    - running average 去看趋势

### details

- vscode/cursor：管理或者 check 代码 diff
- 重视数据/任务/rewards，对数据/任务/rewards要有扎实的分析
    - category，难度，多样性；
    - 充分地 benchmark 不同 size，闭源模型的表现；
- verl/recipe
    - "版本对齐"排第一位
    - 调版本（跑起来） -> 调训练健康度（跑得对） -> 调训练吞吐（跑得快）

#### off-policy / rollout-train mismatch mitigation

一个 RL step 里有两套引擎算同一批 token 的 log-prob:

- rollout:vLLM/sglang(paged KV、融合核、可能 FP8)采样生成
- train:FSDP/Megatron 前向

同样的 token,两套引擎因为 kernel/精度/attention 实现不同,算出的 log-prob 不一样。而 PPO/GRPO 的重要性比值假设"采样策略 = 训练策略",这个 gap 就破坏了 on-policy 假设 —— 这就是 mismatch(引擎内的 off-policy)。MoE 更严重,因为两套引擎还可能把同一个 token 路由到不同的专家。

---

verl 的四层设计

- 用训练引擎重算 old_log_prob(默认常开,第一道防线)
    - `calculate_log_probs=True`。verl 不拿 vLLM 的 log-prob 当 PPO 的分母,而是用训练前向重算一遍 old_log_prob。于是重要性比值是"当前策略 vs 重算的旧策略",两者同引擎,把引擎差异从比值里消掉。vLLM 的 log-prob 只留作监控(②)和可选修正(③)。

#### parameters

- actor_rollout_ref.actor.fsdp_config：“计算低精度、更新高精度”
    - model_dtype: fp32 (default)
        - model_dtype 决定 Hugging Face 权重加载后、FSDP 包装前的“持久参数底座”，也就是 optimizer 最终更新的底座；
    - dtype: bfloat16 (default)：前向/反向主要用 BF16
        - FSDP mixed-precision
        - optimizer 更新的是 FP32 持久参数。
- slime
    - Megatron Distributed Optimizer 明确维护 FP32 master weights、FP32 main grads 和 FP32 Adam moments。

```
FP16 : 1 sign | 5 exponent | 10 fraction
BF16 : 1 sign | 8 exponent |  7 fraction
FP32 : 1 sign | 8 exponent | 23 fraction
```
- FP16 指数范围小，梯度容易下溢（underflow）；
    - 一个接近零的非零数，因为小于当前浮点格式所能表示的最小数，被舍入成零
- BF16 的指数位数和 FP32 相同，动态范围大；
- BF16 的主要短板是尾数精度，而不是动态范围。
    - 参数更新被吸收

```python
if mixed_precision_config is not None:
    param_dtype = mixed_precision_config.get("param_dtype", "bf16")
    reduce_dtype = mixed_precision_config.get("reduce_dtype", "fp32")
    buffer_dtype = mixed_precision_config.get("buffer_dtype", "fp32")
else:
    param_dtype = torch.bfloat16
    reduce_dtype = torch.float32
    buffer_dtype = torch.float32
```

```mermaid
flowchart LR
    A["FP32 sharded parameter"] --> B["All-gather + cast to BF16"]
    B --> C["BF16 forward / activations"]
    C --> D["FP32 PPO loss"]
    D --> E["Backward through BF16 compute graph"]
    E --> F["FP32 reduce-scatter"]
    F --> G["FP32 sharded gradient"]
    G --> H["Grad clip"]
    H --> I["FP32 AdamW state and update"]
    I --> A
```

```mermaid
flowchart LR
    A["BF16 sharded parameter"] --> B["All-gather：仍为 BF16"]
    B --> C["BF16 forward / activations"]
    C --> D["通常为 FP32 的 PPO loss"]
    D --> E["Backward through BF16 compute graph"]
    E --> F["FP32 reduce-scatter"]
    F --> G["Cast back to BF16 sharded gradient"]
    G --> H["Grad clip"]
    H --> I["BF16 AdamW state + BF16 parameter update"]
    I --> A
```

- FP32 reduce 只能提高梯度求和精度，不能恢复前面 BF16 计算已经损失的尾数信息。

In [3]:
import torch

x = torch.tensor(0.1, dtype=torch.bfloat16)
positive_inf = torch.tensor(float("inf"), dtype=torch.bfloat16)

next_x = torch.nextafter(x, positive_inf)
ulp = next_x.float() - x.float()

print("stored x:", x.float().item())
print("next x:  ", next_x.float().item())
# 当前浮点格式两个相邻可表示数之间的距离
print("ULP:     ", ulp.item())

stored x: 0.10009765625
next x:   0.1005859375
ULP:      0.00048828125


In [4]:
0.1005859375 - 0.10009765625

0.00048828125

#### OOM

- CPU OOM / GPU OOM
- vLLM cumem_allocator OOM
    - vLLM wake_up
- FSDP 靠 use_fused_kernels（不物化 logits）

### analysis

> 当效果不好，不符合预期时；

- 回归data，分析数据，数据/难度多样性分布等
    - 原始数据，reward function
    - rollout.jsonl
        - rollout rewards/trajectory group 多样性，方差，那是强化信号的来源和动力